# Coverage-Constrained Price Range / 覆盖率约束价格区间研究

这个 notebook 用来回答一个更直接的问题：

> 在给定 holding period 的情况下，怎样的区间能在尽量不出区间的前提下，做到尽量窄？

这里的 `interval=1d/4h` 只是 data sampling frequency，不是最终要优化的 `price range / 价格区间`。


In [ ]:
import pandas as pd

from app.research import CoverageStudyRequest, plot_coverage_frontier, rename_for_display, run_coverage_study

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)


## 单个 holding period

先看一个固定 holding period，例如 30 天。


In [ ]:
request = CoverageStudyRequest(
    pair="BTC/USDC",
    interval="1d",
    days=730,
    holding_days=30,
    coverage_target_pct=90,
)
result = run_coverage_study(request)
rename_for_display(pd.DataFrame([result.best_interval]))


In [ ]:
rename_for_display(result.frontier)


In [ ]:
plot_coverage_frontier(result.frontier, title="BTC/USDC 30d coverage vs range width frontier")


## 比较多个 holding period

这是这个 notebook 最重要的部分：看 30 天、60 天、180 天、365 天时，最优区间会怎么变化。


In [ ]:
holding_days_list = [30, 60, 180, 365]
results = []
for holding_days in holding_days_list:
    study = run_coverage_study(
        CoverageStudyRequest(
            pair="BTC/USDC",
            interval="1d",
            days=730,
            holding_days=holding_days,
            coverage_target_pct=90,
        )
    )
    row = dict(study.best_interval)
    row["holding_days"] = holding_days
    results.append(row)

comparison = pd.DataFrame(results).sort_values("holding_days")
rename_for_display(comparison[[
    "holding_days",
    "coverage_target_pct",
    "achieved_coverage_pct",
    "lower_bound_pct",
    "upper_bound_pct",
    "width_pct",
    "center_offset_pct",
    "out_of_range_pct",
    "current_lower_price",
    "current_upper_price",
]])


In [ ]:
ax = comparison.plot(x="holding_days", y="width_pct", marker="o", figsize=(8, 4.5), title="Holding period vs range width")
ax.set_xlabel("holding days / 持有期天数")
ax.set_ylabel("range width (%) / 区间宽度(%)")
